In [2]:
import duckdb

con = duckdb.connect('data/coffee.duckdb')
chunks = con.execute('select doc_id, chunk_id, title, content from chunks.chunks').df().to_dict('records')

In [3]:
from minsearch import Index

def build_index(chunks):
    index = Index(
        text_fields=['title', 'content'],
        keyword_fields=['doc_id']
    )
    index.fit(chunks)
    return index

In [16]:
def search(query):
    boost = {}

    results = index.search(
        query=query,
        filter_dict={},
        boost_dict=boost,
        num_results=4
    )

    return results

In [19]:
prompt_template = """
You're a coffee assistant. Answer the QUESTION based on the CONTEXT from our coffee database.
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: {question}

CONTEXT:
{context}
""".strip()

In [22]:
entry_template = """
doc_id: {doc_id}
chunk_id: {chunk_id}
title: {title}
content: {content}
""".strip()

In [24]:
def build_prompt(query, search_results):
    context = ""

    for doc in search_results:
        context = context + entry_template.format(**doc) + "\n\n"

    prompt = prompt_template.format(question=query, context=context).strip()
    return prompt

In [31]:
def llm(prompt, model="gpt-4o-mini"):
    response = client.chat.completions.create(
        model=model, messages=[{"role": "user", "content": prompt}]
    )

    answer = response.choices[0].message.content

    token_stats = {
        "prompt_tokens": response.usage.prompt_tokens,
        "completion_tokens": response.usage.completion_tokens,
        "total_tokens": response.usage.total_tokens,
    }

    return answer, token_stats

In [38]:
def rag(query, model="gpt-4o-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer, token_stats = llm(prompt, model=model)

    answer_data = {
        "answer": answer,
        "model_used": model,
        "prompt_tokens": token_stats["prompt_tokens"],
        "completion_tokens": token_stats["completion_tokens"],
        "total_tokens": token_stats["total_tokens"],
    }
    return answer_data

In [48]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()

index = build_index(chunks)
query = "Why do people drink coffee?"
answer_data = rag(query)

In [49]:
answer_data

{'answer': 'People drink coffee for various reasons, including its caffeine content, which provides a stimulating effect. Regular coffee consumption has been associated with potential benefits such as improved cardiovascular health and a reduced risk of type 2 diabetes and stroke. Additionally, some people enjoy coffee for its taste, cultural significance, and convenience, as seen with products like instant coffee. Coffee also has laxative effects for some individuals, which can be another reason for its consumption.',
 'model_used': 'gpt-4o-mini',
 'prompt_tokens': 1775,
 'completion_tokens': 87,
 'total_tokens': 1862}